In [14]:
# Basic libraries
import pandas as pd
import numpy as np
import re

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# ML preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


In [15]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
pip install openpyxl


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:

df=pd.read_csv("dataset.csv")



In [35]:
df.head()

,filename,text_content,labels,Tokens,Cleaned_Text
0,10425.txt,messageid javamailevansthyme\ndate mon oct ...,"1,1,1\n2,6,1\n2,13,1\n3,3,1\n","[messageid, javamailevansthym, date, mon, oct,...",messageid javamailevansthym date mon oct pdt s...
1,106296.txt,messageid javamailevansthyme\ndate mon mar ...,"1,1,2\n3,6,2\n4,10,2\n","[messageid, javamailevansthym, date, mon, mar,...",messageid javamailevansthym date mon mar pst d...
2,106298.txt,messageid javamailevansthyme\ndate tue mar ...,"1,1,1\n1,6,1\n2,1,1\n2,2,2\n3,6,2\n3,7,1\n4,10...","[messageid, javamailevansthym, date, tue, mar,...",messageid javamailevansthym date tue mar pst d...
3,106588.txt,messageid javamailevansthyme\ndate tue mar ...,"1,1,2\n2,1,2\n2,2,2\n3,6,2\n3,10,2\n","[messageid, javamailevansthym, date, tue, mar,...",messageid javamailevansthym date tue mar pst d...
4,106590.txt,messageid javamailevansthyme\ndate mon mar ...,"1,1,1\n3,6,1\n3,10,1\n4,10,1\n","[messageid, javamailevansthym, date, mon, mar,...",messageid javamailevansthym date mon mar pst d...


In [19]:
df.columns

Index(['filename', 'text_content', 'labels'], dtype='object')

In [20]:
df.shape

(834, 3)

In [21]:
df.isnull().sum()

filename        0
text_content    0
labels          0
dtype: int64

In [22]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [23]:
# Convert Text to Lowercase
df["text_content"] = df["text_content"].str.lower()


In [25]:
# Remove Special Characters & Numbers
def remove_special(text):
    return re.sub(r'[^a-zA-Z\s]', '', text)

df["text_content"] = df["text_content"].apply(remove_special)


In [28]:
# Tokenization
df["Tokens"] = df["text_content"].apply(word_tokenize)


In [29]:
# Remove Stopwords
stop_words = set(stopwords.words("english"))

df["Tokens"] = df["Tokens"].apply(
    lambda words: [word for word in words if word not in stop_words]
)


In [30]:
# Apply Stemming
stemmer = PorterStemmer()

df["Tokens"] = df["Tokens"].apply(
    lambda words: [stemmer.stem(word) for word in words]
)


In [34]:
# Join Tokens Back to Sentence
df["Cleaned_Text"] = df["Tokens"].apply(lambda words: " ".join(words))


In [36]:
# Split Category into Levels
df[["Level1", "Level2", "Level3"]] = df["labels"].str.split('\n').str[0].str.split(",", expand=True)


In [37]:
# Convert Category to Numerical Labels
df["Level1"] = df["Level1"].astype(int)
df["Level2"] = df["Level2"].astype(int)
df["Level3"] = df["Level3"].astype(int)


In [38]:
# Encode Labels (For ML)
le = LabelEncoder()
df["Label"] = le.fit_transform(df["Level3"])

y = df["Label"]


In [39]:
# Convert Text to Numerical Features (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df["Cleaned_Text"])

print("Feature matrix shape:", X.shape)


Feature matrix shape: (834, 5000)


In [40]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)


Training data shape: (667, 5000)
Testing data shape: (167, 5000)


In [41]:
# Save Final Preprocessed Dataset
df.to_excel("Final_Preprocessed_Dataset.xlsx", index=False)
